In [ ]:
import module
await module.load("fsl/6.0.7.16")

In [ ]:
import pathlib, tempfile, subprocess
import nibabel as nib
import numpy as np
work = pathlib.Path(tempfile.mkdtemp(prefix="neurodesktop-bet-phantom-"))
grid = np.indices((64, 64, 64), dtype=float)
radius = np.sqrt(np.sum((grid - 31.5) ** 2, axis=0))
head = np.where(radius <= 20, 100 - radius, np.where((radius >= 23) & (radius <= 25), 30, 0)).astype(np.float32)
nib.save(nib.Nifti1Image(head, np.diag([3, 3, 3, 1])), work / "head.nii.gz")
subprocess.run(["bet", str(work / "head.nii.gz"), str(work / "brain.nii.gz"), "-m"], check=True, timeout=120)

In [ ]:
brain = nib.load(work / "brain.nii.gz").get_fdata()
mask = nib.load(work / "brain_mask.nii.gz").get_fdata()
assert np.isin(mask, [0, 1]).all(), "mask must be binary"
assert np.all(mask[28:36, 28:36, 28:36] == 1), "central brain cube must be retained"
foreground_retention = float(mask[radius <= 18].mean())
assert foreground_retention >= 0.98, "known inner brain must be retained"
outside_fraction = float(mask[radius >= 23].mean())
assert outside_fraction <= 0.001, "known outer skull and background must be excluded"
volume_ratio = float(mask.sum() / np.count_nonzero(radius <= 20))
assert 0.85 <= volume_ratio <= 1.25, "brain mask must match the known spherical anatomy"
assert not np.any(mask[radius > 21.5]), "mask must stay inside the voxel boundary tolerance"
assert np.allclose(brain, head * mask), "brain values must match the retained input"
import json, shutil
print(json.dumps({"foreground_retention": foreground_retention, "outside_fraction": outside_fraction, "volume_ratio": volume_ratio, "mask_voxels": int(mask.sum())}, sort_keys=True))
shutil.rmtree(work)